### 

In [ ]:
import yfinance as yf
import pandas as pd
from pymongo import MongoClient
from datetime import datetime
import time

# Paste your Atlas connection string here
MONGO_URI = "mongodb+srv://<user>:<password>@stock-data.cid0cbn.mongodb.net/?appName=stock-data"
client = MongoClient(MONGO_URI)
db = client["stock_project"]
collection = db["tech_stocks"]

# Compound unique index — prevents duplicates if you re-run ingestion
collection.create_index([("ticker", 1), ("date", 1)], unique=True)
print("Connected and index created.")

Connected and index created.


In [3]:
TICKERS = [
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "NVDA",  # NVIDIA
    "GOOGL", # Alphabet
    "META",  # Meta
    "AMZN",  # Amazon
    "AMD",   # AMD
    "INTC",  # Intel
    "CRM",   # Salesforce
    "ORCL",  # Oracle
    "QCOM",  # Qualcomm
    "ADBE",  # Adobe
    "TSLA",  # Tesla
]

START = "2015-01-01"
END   = "2024-12-31"

In [4]:
failed = []

for ticker in TICKERS:
    try:
        # Download from Yahoo Finance
        raw = yf.download(ticker, start=START, end=END,
                          auto_adjust=True, progress=False)

        if raw.empty:
            print(f"✗ {ticker}: no data returned")
            failed.append(ticker)
            continue

        # Flatten multi-level columns yfinance sometimes returns
        raw.columns = [c[0] if isinstance(c, tuple) else c for c in raw.columns]
        df = raw.reset_index()

        # Engineer the binary label
        df["next_close"] = df["Close"].shift(-1)
        df["label"] = (df["next_close"] > df["Close"]).astype(int)
        df = df.dropna(subset=["next_close"])  # drop last row (no next day)

        # Build one document per trading day
        docs = []
        for _, row in df.iterrows():
            docs.append({
                "ticker":     ticker,
                "date":       row["Date"].to_pydatetime(),
                "open":       round(float(row["Open"]),  4),
                "high":       round(float(row["High"]),  4),
                "low":        round(float(row["Low"]),   4),
                "close":      round(float(row["Close"]), 4),
                "volume":     int(row["Volume"]),
                "next_close": round(float(row["next_close"]), 4),
                "label":      int(row["label"]),  # 1 = up next day, 0 = flat/down
            })

        # ordered=False means a duplicate won't crash the whole batch
        result = collection.insert_many(docs, ordered=False)
        print(f"✓ {ticker}: {len(result.inserted_ids)} documents inserted")

    except Exception as e:
        print(f"✗ {ticker}: {e}")
        failed.append(ticker)

    time.sleep(0.5)  # avoid hammering Yahoo's servers

print(f"\nFinished. Failed: {failed}")

✓ AAPL: 2514 documents inserted
✓ MSFT: 2514 documents inserted
✓ NVDA: 2514 documents inserted
✓ GOOGL: 2514 documents inserted
✓ META: 2514 documents inserted
✓ AMZN: 2514 documents inserted
✓ AMD: 2514 documents inserted
✓ INTC: 2514 documents inserted
✓ CRM: 2514 documents inserted
✓ ORCL: 2514 documents inserted
✓ QCOM: 2514 documents inserted
✓ ADBE: 2514 documents inserted
✓ TSLA: 2514 documents inserted

Finished. Failed: []


In [5]:
total = collection.count_documents({})
by_ticker = collection.aggregate([
    {"$group": {"_id": "$ticker", "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
])

print(f"Total documents: {total}\n")
for doc in by_ticker:
    print(f"  {doc['_id']}: {doc['count']} days")

Total documents: 32682

  AAPL: 2514 days
  ADBE: 2514 days
  AMD: 2514 days
  AMZN: 2514 days
  CRM: 2514 days
  GOOGL: 2514 days
  INTC: 2514 days
  META: 2514 days
  MSFT: 2514 days
  NVDA: 2514 days
  ORCL: 2514 days
  QCOM: 2514 days
  TSLA: 2514 days


In [6]:
label_dist = collection.aggregate([
    {"$group": {"_id": "$label", "count": {"$sum": 1}}}
])
for doc in label_dist:
    label = "Up" if doc["_id"] == 1 else "Down/Flat"
    pct = doc["count"] / 32682 * 100
    print(f"  {label}: {doc['count']} ({pct:.1f}%)")

  Down/Flat: 15437 (47.2%)
  Up: 17245 (52.8%)


In [7]:
stats = collection.aggregate([
    {
        "$group": {
            "_id": None,
            "open_mean":    {"$avg": "$open"},
            "open_std":     {"$stdDevSamp": "$open"},
            "open_min":     {"$min": "$open"},
            "open_max":     {"$max": "$open"},
            "high_mean":    {"$avg": "$high"},
            "high_std":     {"$stdDevSamp": "$high"},
            "high_min":     {"$min": "$high"},
            "high_max":     {"$max": "$high"},
            "low_mean":     {"$avg": "$low"},
            "low_std":      {"$stdDevSamp": "$low"},
            "low_min":      {"$min": "$low"},
            "low_max":      {"$max": "$low"},
            "close_mean":   {"$avg": "$close"},
            "close_std":    {"$stdDevSamp": "$close"},
            "close_min":    {"$min": "$close"},
            "close_max":    {"$max": "$close"},
            "volume_mean":  {"$avg": "$volume"},
            "volume_std":   {"$stdDevSamp": "$volume"},
            "volume_min":   {"$min": "$volume"},
            "volume_max":   {"$max": "$volume"},
        }
    }
])

for doc in stats:
    for k, v in doc.items():
        if k != "_id":
            print(f"{k}: {v:,.4f}")

open_mean: 118.4983
open_std: 116.7623
open_min: 0.4636
open_max: 696.2800
high_mean: 120.0036
high_std: 118.1983
high_min: 0.4679
high_max: 699.5400
low_mean: 116.9453
low_std: 115.2295
low_min: 0.4544
low_max: 678.9100
close_mean: 118.5181
close_std: 116.7559
close_min: 0.4592
close_max: 688.3700
volume_mean: 75,657,607.5301
volume_std: 142,140,914.4941
volume_min: 0.0000
volume_max: 3,692,928,000.0000
